<a href="https://colab.research.google.com/github/sergiocostaifes/PPCOMP_DM/blob/main/notebooks/05_feature_engineering_FULL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB05_FULL — Engenharia de Atributos Temporais por Célula

## 1. Contexto

O **NB05_FULL** recebe as séries temporais pontuadas e os episódios críticos produzidos pelo **NB04_FULL** sob a política `TRAIN_M2S_BY_CELL`. As oito células Borg (`a`–`h`) são tratadas como unidades experimentais independentes, preservando a ordem temporal e os parâmetros herdados da etapa anterior.

## 2. Objetivo

Esta etapa:

- valida a correspondência entre a série pontuada e os episódios críticos;
- preserva `event_FAIL_count` no artefato model-facing para proveniência e auditoria;
- confirma a igualdade exata entre `event_FAIL_count` e `n_failed`;
- utiliza `n_failed` como representação única da contagem de falhas nos vetores preditores;
- constrói atributos temporais causais separadamente por célula;
- remove somente as janelas iniciais sem histórico suficiente para as defasagens;
- persiste artefatos por célula e uma consolidação rastreável.

## 3. Entradas

Para cada `cell_id`, são esperados:

```text
04-reports/99_FULL_downstream/04_FULL_episodes/cell_<id>/
  04_FULL_window_5min_series_scored_TRAIN_M2S_cell_<id>.parquet
  04_FULL_episodes_TRAIN_M2S_cell_<id>.parquet
  04_FULL_detect_episodes_summary_cell_<id>.json  # quando disponível
```

## 4. Saídas

Para cada célula:

```text
04-reports/99_FULL_downstream/05_FULL_features/cell_<id>/
  05_FULL_window_5min_features_TRAIN_M2S_cell_<id>.parquet
  05_FULL_feature_engineering_summary_cell_<id>.json
```

Na pasta `aggregate/`:

```text
05_FULL_feature_summary_by_cell.csv
05_FULL_window_5min_features_active_cells.parquet
05_FULL_feature_engineering_summary.json
05_FULL_feature_sets.json
05_FULL_delta_vs_canonical.csv
05_FULL_artifact_manifest_sha256.csv
```

O nome legado `05_FULL_delta_vs_canonical.csv` é preservado como contrato de artefato; seu conteúdo descreve a comparação com o notebook de referência do protótipo Kaggle.

## 5. Conjuntos de atributos

O conjunto bruto ativo é:

```text
raw_core = [
  fail_rate, n_events, n_failed, n_machines,
  n_collections, event_LOST_count
]
```

O conjunto `temporal_core` acrescenta:

```text
lag_1, lag_2, lag_3, rolling_mean_1h,
rolling_std_1h, pct_change, zscore_expanding
```

Assim, `raw_core` contém **6 atributos** e `temporal_core` contém **13 atributos**. `event_FAIL_count` permanece disponível no Parquet, mas não integra qualquer vetor de preditores.

## 6. Disciplina temporal

As defasagens, estatísticas móveis e estatística expansiva são calculadas dentro de cada célula. O agregado final é uma concatenação rastreável das oito séries e não deve ser interpretado como uma única série temporal.

## 7. Travas metodológicas

- a criticidade não é redefinida;
- os episódios não são redetectados;
- os limiares não são recalculados;
- células distintas não são combinadas para o cálculo de atributos;
- `event_FAIL_count` não pode integrar `raw_core` ou `temporal_core`;
- alvo, estado, rótulos e informações futuras não podem integrar os preditores;
- duplicidades exatas não justificadas entre atributos ativos interrompem a execução.

## 8. Validações obrigatórias

A execução verifica:

1. igualdade exata entre `n_failed` e `event_FAIL_count`;
2. cardinalidades 6 e 13 dos conjuntos ativos;
3. unicidade dos nomes dos atributos;
4. ausência da coluna redundante nos preditores;
5. ausência de atributos proibidos;
6. ausência de duplicidade exata entre colunas ativas;
7. continuidade temporal, unicidade de `bucket_id` e identidade da célula;
8. consistência de `fail_rate = n_failed / n_events`;
9. ausência de valores nulos ou infinitos após a engenharia;
10. equivalência entre o agregado e os artefatos por célula.

## 9. Critério de aceite

O notebook está apto a alimentar o **NB06_FULL** somente quando todas as células ativas concluírem com `status = OK`, as travas de redundância forem aprovadas e o manifesto SHA-256 for regenerado ao final da etapa.


In [ ]:
# ============================================================
# NB05_FULL — Engenharia de Atributos por Célula
# Pipeline PPCOMP_DM — Ramo _FULL
#
# Escopo:
# - Ler, por célula, a série scored produzida pelo NB04_FULL
# - Ler, por célula, os episódios críticos oficiais do NB04_FULL
# - Validar consistência entre série scored e episódios
# - Representar explicitamente janelas vazias
# - Imputar campos médios ausentes apenas nas janelas sem eventos
# - Criar atributos temporais causais baseados em fail_rate
# - Remover apenas as linhas iniciais sem lags suficientes por célula
# - Persistir artefatos por célula e agregados
# - Persistir summary, feature_sets, delta auditável e manifesto SHA-256
#
# Importante:
# - Este notebook não redefine criticidade
# - Este notebook não redetecta episódios
# - Este notebook não recalcula limiares
# - Este notebook não concatena células para cálculo de features temporais
# - Cada cell_id é tratado como réplica independente do experimento _FULL
# ============================================================

# ─────────────────────────────────────────────────────────────
# BLOCO 0 — Bootstrap do ambiente
# ─────────────────────────────────────────────────────────────

from pathlib import Path
import os
import sys
import subprocess
import importlib
import random
import json
import hashlib
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def log(msg: str) -> None:
    """Imprime mensagens padronizadas desta etapa do pipeline."""
    print(f"[NB05_FULL_feature_engineering] {msg}")

# Monta o Google Drive apenas quando necessário.
if not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        log(f"Aviso: Google Drive não montado automaticamente ({e}).")
else:
    print("[Bootstrap] Google Drive já montado.")

# Repositório local no Google Drive, preservando o padrão dos notebooks do protótipo Kaggle.
REPO_DIR = Path("/content/drive/MyDrive/Mestrado/PPCOMP_DM")
GITHUB_REPO = "https://github.com/sergiocostaifes/PPCOMP_DM.git"

if not REPO_DIR.exists():
    try:
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        print(f"[Bootstrap] Clonando repositório em: {REPO_DIR}")
        subprocess.run(["git", "clone", GITHUB_REPO, str(REPO_DIR)], check=True)
    except Exception as e:
        log(f"Aviso: não foi possível clonar o repositório automaticamente ({e}).")
else:
    try:
        print("[Bootstrap] Atualizando repositório (git pull)...")
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)
    except Exception as e:
        print("[Bootstrap] Aviso: não foi possível atualizar:", e)

if REPO_DIR.exists():
    os.chdir(str(REPO_DIR))
    repo_str = str(REPO_DIR)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)
    importlib.invalidate_caches()

print("[Bootstrap] CWD =", os.getcwd())

# ─────────────────────────────────────────────────────────────
# BLOCO 1 — Parâmetros, caminhos e constantes congeladas
# ─────────────────────────────────────────────────────────────

DRIVE_ROOT = Path(os.environ.get("DRIVE_ROOT", "/content/drive/MyDrive/Mestrado"))

FULL_MODEL_FACING_PARQUET = (
    DRIVE_ROOT
    / "02-datasets"
    / "99-full"
    / "03-model-facing"
    / "window_5min_series_allcells_model_facing_000000000000.parquet"
)

FULL_REPORTS_DIR = DRIVE_ROOT / "04-reports" / "99_FULL_downstream"
STAGE04_DIR = FULL_REPORTS_DIR / "04_FULL_episodes"
STAGE05_DIR = FULL_REPORTS_DIR / "05_FULL_features"
AGGREGATE_DIR = STAGE05_DIR / "aggregate"

SCENARIO_LABEL = "TRAIN_M2S"
THRESHOLD_POLICY = "TRAIN_M2S_BY_CELL"

FULL_CELLS = list("abcdefgh")

# Edite aqui para piloto, se necessário. Para execução oficial, manter FULL_CELLS.
# Também é possível definir variável de ambiente ACTIVE_CELLS="a,b,c".
ACTIVE_CELLS_ENV = os.environ.get("ACTIVE_CELLS", "").strip()
if ACTIVE_CELLS_ENV:
    ACTIVE_CELLS = [c.strip().lower() for c in ACTIVE_CELLS_ENV.split(",") if c.strip()]
else:
    ACTIVE_CELLS = FULL_CELLS

# Parâmetros herdados do cenário de referência.
WINDOW_MINUTES = 5
K_BEFORE_AFTER = 24
HORIZON_H = 12
PERSISTENCE_P = 1

RAW_CORE = [
    "fail_rate",
    "n_events",
    "n_failed",
    "n_machines",
    "n_collections",
    "event_LOST_count",
]

FEATURES_CREATED = [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]

LAG_COLS = ["lag_1", "lag_2", "lag_3"]

TEMPORAL_CORE = RAW_CORE + [
    "lag_1",
    "lag_2",
    "lag_3",
    "rolling_mean_1h",
    "rolling_std_1h",
    "pct_change",
    "zscore_expanding",
]

assert "event_FAIL_count" not in RAW_CORE
assert "event_FAIL_count" not in TEMPORAL_CORE
assert len(RAW_CORE) == 6, "raw_core deve conter 6 atributos."
assert len(TEMPORAL_CORE) == 13, "temporal_core deve conter 13 atributos."
assert len(RAW_CORE) == len(set(RAW_CORE)), "raw_core contém nomes duplicados."
assert len(TEMPORAL_CORE) == len(set(TEMPORAL_CORE)), "temporal_core contém nomes duplicados."
assert set(ACTIVE_CELLS).issubset(set(FULL_CELLS)), f"ACTIVE_CELLS inválido: {ACTIVE_CELLS}"
assert len(ACTIVE_CELLS) == len(set(ACTIVE_CELLS)), "ACTIVE_CELLS contém células duplicadas."
assert FULL_MODEL_FACING_PARQUET.name != "window_5min_series.parquet"
assert "03-features" not in str(FULL_REPORTS_DIR), "NB05_FULL não pode escrever em 03-features."

for path in [STAGE05_DIR, AGGREGATE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

for cell_id in ACTIVE_CELLS:
    (STAGE05_DIR / f"cell_{cell_id}").mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT =", DRIVE_ROOT)
print("FULL_MODEL_FACING_PARQUET =", FULL_MODEL_FACING_PARQUET)
print("STAGE04_DIR =", STAGE04_DIR)
print("STAGE05_DIR =", STAGE05_DIR)
print("AGGREGATE_DIR =", AGGREGATE_DIR)
print("ACTIVE_CELLS =", ACTIVE_CELLS)
print("RAW_CORE =", RAW_CORE)
print("TEMPORAL_CORE =", TEMPORAL_CORE)

# ─────────────────────────────────────────────────────────────
# BLOCO 2 — Funções utilitárias
# ─────────────────────────────────────────────────────────────

def read_json_if_exists(path: Path) -> dict:
    """Lê arquivo JSON se existir."""
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def first_not_none(*values):
    """Retorna o primeiro valor não nulo."""
    for value in values:
        if value is not None:
            return value
    return None

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    """Calcula SHA-256 de um arquivo."""
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def build_manifest(stage_dir: Path) -> pd.DataFrame:
    """Constrói manifesto SHA-256 dos artefatos da etapa."""
    rows = []
    for path in sorted(stage_dir.rglob("*")):
        if path.is_file():
            rows.append(
                {
                    "relative_path": str(path.relative_to(stage_dir)).replace("\\", "/"),
                    "file_size_bytes": int(path.stat().st_size),
                    "sha256": sha256_file(path),
                }
            )
    return pd.DataFrame(rows)

def detect_critical_blocks_from_scored(df_cell: pd.DataFrame) -> int:
    """Conta blocos contíguos de is_critical=True sem redetectar episódios."""
    flags = df_cell["is_critical"].astype(bool)
    starts = flags & ~flags.shift(1, fill_value=False)
    return int(starts.sum())

def cell_input_paths(cell_id: str) -> dict:
    """Retorna caminhos de entrada esperados para uma célula."""
    cell_dir = STAGE04_DIR / f"cell_{cell_id}"
    return {
        "series": cell_dir / f"04_FULL_window_5min_series_scored_{SCENARIO_LABEL}_cell_{cell_id}.parquet",
        "episodes": cell_dir / f"04_FULL_episodes_{SCENARIO_LABEL}_cell_{cell_id}.parquet",
        "summary": cell_dir / f"04_FULL_detect_episodes_summary_cell_{cell_id}.json",
    }

def cell_output_paths(cell_id: str) -> dict:
    """Retorna caminhos de saída da etapa para uma célula."""
    cell_dir = STAGE05_DIR / f"cell_{cell_id}"
    return {
        "features": cell_dir / f"05_FULL_window_5min_features_{SCENARIO_LABEL}_cell_{cell_id}.parquet",
        "summary": cell_dir / f"05_FULL_feature_engineering_summary_cell_{cell_id}.json",
    }

def validate_required_columns(df: pd.DataFrame, episodes: pd.DataFrame, cell_id: str) -> None:
    """Valida colunas mínimas e o schema esperado para a etapa."""
    required_series_cols = {
        "cell_id",
        "bucket_id",
        "bucket_start_us",
        "n_events",
        "n_failed",
        "fail_rate",
        "is_critical",
        "threshold_train_m2s",
        "threshold_train_mu",
        "threshold_train_sigma",
    }

    missing_series = required_series_cols - set(df.columns)
    assert not missing_series, f"Célula {cell_id}: colunas ausentes na série scored: {missing_series}"

    missing_raw_core = set(RAW_CORE) - set(df.columns)
    assert not missing_raw_core, (
        f"Célula {cell_id}: atributos de raw_core ausentes na origem FULL: {missing_raw_core}"
    )
    assert "event_FAIL_count" in df.columns, (
        f"Célula {cell_id}: event_FAIL_count deve permanecer no artefato para proveniência."
    )

    forbidden_cols = {
        "fail_rate_all_events",
        "n_events_all",
        "type_5_FAIL_count",
        "type_8_LOST_count",
    }
    present_forbidden = sorted(forbidden_cols & set(df.columns))
    assert not present_forbidden, f"Célula {cell_id}: colunas depreciadas/proibidas presentes: {present_forbidden}"

    required_episode_cols = {"cell_id", "start_bucket", "end_bucket", "duration_windows"}
    missing_ep = required_episode_cols - set(episodes.columns)
    assert not missing_ep, f"Célula {cell_id}: colunas ausentes nos episódios: {missing_ep}"

def validate_cell_identity(df: pd.DataFrame, episodes: pd.DataFrame, cell_id: str) -> None:
    """Garante que a célula processada não foi misturada com outras."""
    assert "cell_id" in df.columns, "Série scored precisa conter cell_id."
    observed_series_cells = sorted(df["cell_id"].dropna().astype(str).unique().tolist())
    assert observed_series_cells == [cell_id], (
        f"Esperado somente cell_id={cell_id} na série; observado={observed_series_cells}"
    )

    observed_episode_cells = sorted(episodes["cell_id"].dropna().astype(str).unique().tolist())
    assert observed_episode_cells == [cell_id], (
        f"Esperado somente cell_id={cell_id} nos episódios; observado={observed_episode_cells}"
    )

def validate_temporal_continuity(df: pd.DataFrame, cell_id: str) -> None:
    """Valida ordenação, unicidade e continuidade temporal por bucket_id."""
    assert df["bucket_id"].is_monotonic_increasing, f"Célula {cell_id}: bucket_id não está ordenado."
    assert df["bucket_id"].is_unique, f"Célula {cell_id}: bucket_id duplicado."

    bucket_diff = df["bucket_id"].diff().dropna()
    n_internal_gaps_gt1 = int((bucket_diff > 1).sum())
    assert n_internal_gaps_gt1 == 0, (
        f"Célula {cell_id}: série scored deveria estar contínua após NB04_FULL."
    )

def validate_fail_rate(df: pd.DataFrame, cell_id: str) -> float:
    """Recalcula fail_rate e retorna diferença absoluta máxima."""
    denom = df["n_events"].replace(0, np.nan)
    recomputed = (df["n_failed"] / denom).fillna(0.0)
    diff = (df["fail_rate"].astype(float) - recomputed.astype(float)).abs()
    max_abs_diff = float(diff.max())
    assert max_abs_diff <= 1e-12, (
        f"Célula {cell_id}: fail_rate divergente do recomputado; max_abs_diff={max_abs_diff}"
    )
    return max_abs_diff

def exact_duplicate_feature_pairs(df: pd.DataFrame, features: list[str]) -> list[tuple[str, str]]:
    """Identifica colunas ativas com valores exatamente iguais."""
    pairs = []
    for i, left in enumerate(features):
        for right in features[i + 1:]:
            if df[left].equals(df[right]):
                pairs.append((left, right))
    return pairs

def process_cell(cell_id: str) -> tuple[pd.DataFrame, dict]:
    """Executa a engenharia de atributos para uma única célula."""
    paths_in = cell_input_paths(cell_id)
    paths_out = cell_output_paths(cell_id)

    assert paths_in["series"].exists(), f"Célula {cell_id}: arquivo não encontrado: {paths_in['series']}"
    assert paths_in["episodes"].exists(), f"Célula {cell_id}: arquivo não encontrado: {paths_in['episodes']}"

    df = pd.read_parquet(paths_in["series"])
    episodes = pd.read_parquet(paths_in["episodes"])
    nb04_summary = read_json_if_exists(paths_in["summary"])

    df = df.sort_values("bucket_id").reset_index(drop=True)
    episodes = episodes.sort_values(["start_bucket", "end_bucket"]).reset_index(drop=True)

    log(f"Célula {cell_id}: shape série scored={df.shape}; episodes={episodes.shape}")

    validate_required_columns(df, episodes, cell_id)

    fail_count_equal = bool(
        df["n_failed"].reset_index(drop=True).equals(
            df["event_FAIL_count"].reset_index(drop=True)
        )
    )
    assert fail_count_equal, (
        f"Célula {cell_id}: n_failed diverge de event_FAIL_count; "
        "a execução foi interrompida antes da engenharia de atributos."
    )

    validate_cell_identity(df, episodes, cell_id)
    validate_temporal_continuity(df, cell_id)
    fail_rate_max_abs_diff = validate_fail_rate(df, cell_id)

    rows_input = int(len(df))
    bucket_id_min = int(df["bucket_id"].min())
    bucket_id_max = int(df["bucket_id"].max())

    critical_windows_input = int(df["is_critical"].sum())
    episodes_detected_input = int(len(episodes))
    critical_windows_from_episodes = int(episodes["duration_windows"].sum())
    critical_blocks_from_series = detect_critical_blocks_from_scored(df)

    assert critical_windows_input == critical_windows_from_episodes, (
        f"Célula {cell_id}: is_critical não fecha com soma de duration_windows."
    )
    assert episodes_detected_input == critical_blocks_from_series, (
        f"Célula {cell_id}: número de episódios não fecha com blocos contíguos críticos."
    )

    official_result = nb04_summary.get("official_result", {})
    expected_critical_windows = first_not_none(
        nb04_summary.get("critical_windows"),
        official_result.get("critical_windows"),
    )
    expected_episodes = first_not_none(
        nb04_summary.get("episodes_detected"),
        official_result.get("episodes_detected"),
    )

    if expected_critical_windows is not None:
        assert critical_windows_input == int(expected_critical_windows), (
            f"Célula {cell_id}: critical_windows difere do summary NB04_FULL."
        )

    if expected_episodes is not None:
        assert episodes_detected_input == int(expected_episodes), (
            f"Célula {cell_id}: episodes_detected difere do summary NB04_FULL."
        )

    # ─────────────────────────────────────────────────────────
    # Tratamento de janelas vazias e imputações locais
    # ─────────────────────────────────────────────────────────

    df["is_empty_window"] = (df["n_events"] == 0).astype("int8")
    empty_windows = int(df["is_empty_window"].sum())

    mean_cols_to_impute = [
        c for c in ["mean_priority", "mean_req_cpus", "mean_req_mem"]
        if c in df.columns
    ]

    na_before_impute = {
        c: int(df[c].isna().sum())
        for c in mean_cols_to_impute
    }

    for c in mean_cols_to_impute:
        df[c] = df[c].fillna(0.0).astype("float32")

    na_after_impute = {
        c: int(df[c].isna().sum())
        for c in mean_cols_to_impute
    }

    # ─────────────────────────────────────────────────────────
    # Construção de features temporais causais por célula
    # ─────────────────────────────────────────────────────────

    base = "fail_rate"

    df["lag_1"] = df[base].shift(1)
    df["lag_2"] = df[base].shift(2)
    df["lag_3"] = df[base].shift(3)

    # Janela móvel de 1 hora: 12 janelas de 5 minutos.
    # Mantém a lógica temporal estabelecida no protótipo Kaggle, incluindo a janela atual.
    W = 12

    df["rolling_mean_1h"] = (
        df[base]
        .rolling(W, min_periods=1)
        .mean()
        .astype("float32")
    )

    df["rolling_std_1h"] = (
        df[base]
        .rolling(W, min_periods=1)
        .std(ddof=0)
        .fillna(0.0)
        .astype("float32")
    )

    df["pct_change"] = df[base].pct_change()
    df["pct_change"] = (
        df["pct_change"]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype("float32")
    )

    # Normalização adaptativa e causal:
    # usa apenas histórico disponível até t-1.
    hist_mean = df[base].shift(1).expanding(min_periods=1).mean()
    hist_std = df[base].shift(1).expanding(min_periods=2).std(ddof=0)

    df["zscore_expanding"] = (
        ((df[base] - hist_mean) / hist_std.replace(0, np.nan))
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .astype("float32")
    )

    # ─────────────────────────────────────────────────────────
    # Limpeza final: remoção mínima por lags
    # ─────────────────────────────────────────────────────────

    rows_before_trim = int(len(df))
    lag_cols = LAG_COLS
    rows_missing_lag = int(df[lag_cols].isna().any(axis=1).sum())

    # Remove apenas as primeiras linhas sem histórico suficiente.
    df = df.dropna(subset=lag_cols).reset_index(drop=True)

    rows_after_trim = int(len(df))
    rows_removed_trim = rows_before_trim - rows_after_trim

    assert rows_removed_trim == rows_missing_lag, (
        f"Célula {cell_id}: trim por lag removeu quantidade inesperada de linhas."
    )
    assert rows_removed_trim == len(LAG_COLS), (
        f"Célula {cell_id}: esperado remover {len(LAG_COLS)} linhas iniciais por lags; observado={rows_removed_trim}"
    )

    for c in lag_cols:
        df[c] = df[c].astype("float32")

    missing_temporal_core = set(TEMPORAL_CORE) - set(df.columns)
    assert not missing_temporal_core, (
        f"Célula {cell_id}: temporal_core incompleto após features: {missing_temporal_core}"
    )

    na_temporal_core = {
        c: int(df[c].isna().sum())
        for c in TEMPORAL_CORE
    }
    assert sum(na_temporal_core.values()) == 0, (
        f"Célula {cell_id}: temporal_core contém NaN: {na_temporal_core}"
    )

    forbidden_predictors = {
        "cell_id", "bucket_id", "bucket_start_us", "is_critical", "state",
        "target", "y", "y_binary", "anticipation_target",
        "threshold_train_m2s", "threshold_train_mu", "threshold_train_sigma",
        "episode_id", "start_bucket", "end_bucket",
    }
    active_features = TEMPORAL_CORE
    present_forbidden_predictors = sorted(set(active_features) & forbidden_predictors)
    assert not present_forbidden_predictors, (
        f"Célula {cell_id}: preditores proibidos presentes: {present_forbidden_predictors}"
    )

    duplicate_feature_pairs = exact_duplicate_feature_pairs(df, active_features)
    assert not duplicate_feature_pairs, (
        f"Célula {cell_id}: duplicidades exatas não justificadas entre preditores: "
        f"{duplicate_feature_pairs}"
    )

    critical_windows_output = int(df["is_critical"].sum())
    critical_ratio_output = float(df["is_critical"].mean())
    numeric_cols_final = df.select_dtypes(include=[np.number]).columns.tolist()

    # Persistência por célula
    df.to_parquet(paths_out["features"], compression="snappy", index=False)

    summary = {
        "notebook": "NB05_FULL",
        "purpose": "feature_engineering_by_cell",
        "cell_id": cell_id,
        "scenario_label": SCENARIO_LABEL,
        "threshold_policy_inherited": THRESHOLD_POLICY,
        "window_minutes": WINDOW_MINUTES,
        "K_before_after": K_BEFORE_AFTER,
        "H_horizon": HORIZON_H,
        "P_persistence": PERSISTENCE_P,
        "input_series_file": str(paths_in["series"]),
        "input_episodes_file": str(paths_in["episodes"]),
        "input_nb04_summary_file": str(paths_in["summary"]) if paths_in["summary"].exists() else None,
        "output_features_file": str(paths_out["features"]),
        "rows_input": int(rows_input),
        "rows_before_trim": int(rows_before_trim),
        "rows_output": int(rows_after_trim),
        "rows_removed_trim": int(rows_removed_trim),
        "bucket_id_min_input": int(bucket_id_min),
        "bucket_id_max_input": int(bucket_id_max),
        "bucket_id_min_output": int(df["bucket_id"].min()),
        "bucket_id_max_output": int(df["bucket_id"].max()),
        "critical_windows_input": int(critical_windows_input),
        "critical_windows_output": int(critical_windows_output),
        "critical_ratio_output": float(critical_ratio_output),
        "episodes_detected_input": int(episodes_detected_input),
        "critical_windows_from_episodes": int(critical_windows_from_episodes),
        "critical_blocks_from_series": int(critical_blocks_from_series),
        "empty_windows": int(empty_windows),
        "fail_rate_max_abs_diff_recomputed": float(fail_rate_max_abs_diff),
        "na_before_impute": na_before_impute,
        "na_after_impute": na_after_impute,
        "base_metric": base,
        "event_FAIL_count_preserved_for_provenance": True,
        "n_failed_equals_event_FAIL_count": fail_count_equal,
        "event_FAIL_count_in_predictors": False,
        "raw_core": RAW_CORE,
        "n_raw_core": int(len(RAW_CORE)),
        "feature_cols_created": FEATURES_CREATED,
        "temporal_core": TEMPORAL_CORE,
        "n_temporal_core": int(len(TEMPORAL_CORE)),
        "exact_duplicate_feature_pairs": duplicate_feature_pairs,
        "n_numeric_cols_final": int(len(numeric_cols_final)),
        "numeric_cols_final": numeric_cols_final,
        "preliminary_modeling_status_from_nb04": first_not_none(
            nb04_summary.get("preliminary_modeling_status"),
            nb04_summary.get("status_modelagem"),
        ),
        "status": "OK",
    }

    paths_out["summary"].write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    log(
        f"Célula {cell_id}: rows {rows_input} -> {rows_after_trim}; "
        f"critical {critical_windows_input} -> {critical_windows_output}; "
        f"features={paths_out['features'].name}"
    )

    return df, summary

# ─────────────────────────────────────────────────────────────
# BLOCO 3 — Execução por célula
# ─────────────────────────────────────────────────────────────

all_features = []
summaries = []

for cell_id in ACTIVE_CELLS:
    log(f"Iniciando célula {cell_id}")
    df_cell_features, summary_cell = process_cell(cell_id)
    all_features.append(df_cell_features)
    summaries.append(summary_cell)

summary_by_cell = pd.DataFrame(summaries).sort_values("cell_id").reset_index(drop=True)

# Validação consolidada sem tratar as células como uma série temporal única.
assert sorted(summary_by_cell["cell_id"].tolist()) == sorted(ACTIVE_CELLS)
assert (summary_by_cell["rows_input"] == 8928).all(), (
    "Cada célula deveria preservar 8.928 janelas de entrada do NB04_FULL."
)
assert (summary_by_cell["rows_removed_trim"] == len(LAG_COLS)).all(), (
    "Cada célula deveria remover apenas 3 linhas iniciais por lag."
)

aggregate_features = (
    pd.concat(all_features, ignore_index=True)
    .sort_values(["cell_id", "bucket_id"])
    .reset_index(drop=True)
)

observed_cells_aggregate = sorted(aggregate_features["cell_id"].astype(str).unique().tolist())
assert observed_cells_aggregate == sorted(ACTIVE_CELLS), (
    f"Aggregate contém células inesperadas: {observed_cells_aggregate}"
)

expected_rows_aggregate = int(summary_by_cell["rows_output"].sum())
assert len(aggregate_features) == expected_rows_aggregate

for cell_id, df_group in aggregate_features.groupby("cell_id", sort=True):
    assert df_group["bucket_id"].is_monotonic_increasing
    assert df_group["bucket_id"].is_unique

# ─────────────────────────────────────────────────────────────
# BLOCO 4 — Artefatos agregados
# ─────────────────────────────────────────────────────────────

SUMMARY_BY_CELL_FILE = AGGREGATE_DIR / "05_FULL_feature_summary_by_cell.csv"
AGG_FEATURES_FILE = AGGREGATE_DIR / "05_FULL_window_5min_features_active_cells.parquet"
FEATURE_SETS_FILE = AGGREGATE_DIR / "05_FULL_feature_sets.json"
SUMMARY_JSON_FILE = AGGREGATE_DIR / "05_FULL_feature_engineering_summary.json"
DELTA_FILE = AGGREGATE_DIR / "05_FULL_delta_vs_canonical.csv"

summary_by_cell.to_csv(SUMMARY_BY_CELL_FILE, index=False)
aggregate_features.to_parquet(AGG_FEATURES_FILE, compression="snappy", index=False)

feature_sets = {
    "notebook": "NB05_FULL",
    "source": "Conjuntos ativos definidos pelo protocolo do experimento FULL",
    "scenario_label": SCENARIO_LABEL,
    "raw_core": RAW_CORE,
    "n_raw_core": len(RAW_CORE),
    "temporal_core": TEMPORAL_CORE,
    "n_temporal_core": len(TEMPORAL_CORE),
    "features_created_in_nb05_full": FEATURES_CREATED,
    "provenance_columns_excluded_from_predictors": ["event_FAIL_count"],
    "notes": [
        "NB05_FULL preserva o schema do Parquet model-facing do experimento FULL.",
        "Features temporais são calculadas independentemente por cell_id.",
        "Aggregate é apenas artefato de conveniência e rastreabilidade, não uma série temporal única."
    ],
}
FEATURE_SETS_FILE.write_text(
    json.dumps(feature_sets, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

aggregate_summary = {
    "notebook": "NB05_FULL",
    "purpose": "feature_engineering_by_cell_full_downstream",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "active_cells": ACTIVE_CELLS,
    "scenario_label": SCENARIO_LABEL,
    "threshold_policy_inherited": THRESHOLD_POLICY,
    "window_minutes": WINDOW_MINUTES,
    "K_before_after": K_BEFORE_AFTER,
    "H_horizon": HORIZON_H,
    "P_persistence": PERSISTENCE_P,
    "input_stage": str(STAGE04_DIR),
    "output_stage": str(STAGE05_DIR),
    "n_cells_processed": int(len(ACTIVE_CELLS)),
    "rows_input_total": int(summary_by_cell["rows_input"].sum()),
    "rows_output_total": int(summary_by_cell["rows_output"].sum()),
    "rows_removed_trim_total": int(summary_by_cell["rows_removed_trim"].sum()),
    "critical_windows_input_total": int(summary_by_cell["critical_windows_input"].sum()),
    "critical_windows_output_total": int(summary_by_cell["critical_windows_output"].sum()),
    "episodes_detected_total": int(summary_by_cell["episodes_detected_input"].sum()),
    "empty_windows_total": int(summary_by_cell["empty_windows"].sum()),
    "raw_core": RAW_CORE,
    "n_raw_core": int(len(RAW_CORE)),
    "event_FAIL_count_preserved_for_provenance": True,
    "event_FAIL_count_in_predictors": False,
    "temporal_core": TEMPORAL_CORE,
    "n_temporal_core": int(len(TEMPORAL_CORE)),
    "aggregate_features_file": str(AGG_FEATURES_FILE),
    "summary_by_cell_file": str(SUMMARY_BY_CELL_FILE),
    "feature_sets_file": str(FEATURE_SETS_FILE),
    "status": "OK",
}
SUMMARY_JSON_FILE.write_text(
    json.dumps(aggregate_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

delta_vs_canonical = pd.DataFrame(
    [
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "I/O_PATH",
            "description": "Entrada passa dos artefatos do protótipo Kaggle para os artefatos do NB04_FULL em 99_FULL_downstream.",
            "justified": True,
        },
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "CELL_LOOP",
            "description": "Execução parametrizada por ACTIVE_CELLS, com processamento independente por cell_id.",
            "justified": True,
        },
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "FULL_DATASET_INPUT",
            "description": "Uso do downstream _FULL derivado do Parquet model-facing do NB99_C.",
            "justified": True,
        },
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "OUTPUT_PREFIX",
            "description": "Todos os artefatos recebem prefixo 05_FULL para impedir sobrescrita dos artefatos do protótipo Kaggle.",
            "justified": True,
        },
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "AGGREGATION_BY_CELL",
            "description": "Criação de pasta aggregate com sumários por célula e concatenação apenas para rastreabilidade.",
            "justified": True,
        },
        {
            "notebook_full": "05_feature_engineering_FULL.ipynb",
            "notebook_reference": "05_feature_engineering.ipynb",
            "change_type": "REPORTING_ONLY",
            "description": "Registro explícito de raw_core com 6 atributos e temporal_core com 13 atributos, sem event_FAIL_count nos preditores.",
            "justified": True,
        },
    ]
)
delta_vs_canonical.to_csv(DELTA_FILE, index=False)

# Manifesto após todos os artefatos da etapa.
MANIFEST_FILE = AGGREGATE_DIR / "05_FULL_artifact_manifest_sha256.csv"
manifest = build_manifest(STAGE05_DIR)
manifest.to_csv(MANIFEST_FILE, index=False)

# ─────────────────────────────────────────────────────────────
# BLOCO 5 — Visualização rápida
# ─────────────────────────────────────────────────────────────

print("\n=== RESUMO FINAL — NB05_FULL ===")
print("ACTIVE_CELLS:", ACTIVE_CELLS)
print("Entrada NB04_FULL:", STAGE04_DIR)
print("Saída NB05_FULL:", STAGE05_DIR)
print("Aggregate:", AGGREGATE_DIR)
print("Scenario label:", SCENARIO_LABEL)
print("Raw core:", len(RAW_CORE), "atributos")
print("Temporal core:", len(TEMPORAL_CORE), "atributos")

print("\n[Resumo por célula]")
display(
    summary_by_cell[
        [
            "cell_id",
            "rows_input",
            "rows_output",
            "rows_removed_trim",
            "critical_windows_input",
            "critical_windows_output",
            "episodes_detected_input",
            "empty_windows",
            "fail_rate_max_abs_diff_recomputed",
            "n_temporal_core",
            "status",
        ]
    ]
)

print("\n[Totais]")
display(
    pd.DataFrame(
        [
            {
                "n_cells_processed": len(ACTIVE_CELLS),
                "rows_input_total": aggregate_summary["rows_input_total"],
                "rows_output_total": aggregate_summary["rows_output_total"],
                "rows_removed_trim_total": aggregate_summary["rows_removed_trim_total"],
                "critical_windows_input_total": aggregate_summary["critical_windows_input_total"],
                "critical_windows_output_total": aggregate_summary["critical_windows_output_total"],
                "episodes_detected_total": aggregate_summary["episodes_detected_total"],
                "empty_windows_total": aggregate_summary["empty_windows_total"],
                "n_manifest_files": int(len(manifest)),
            }
        ]
    )
)

print("\n[Head aggregate features]")
display(aggregate_features.head())

print("\nArtefatos aggregate:")
for p in [
    SUMMARY_BY_CELL_FILE,
    AGG_FEATURES_FILE,
    FEATURE_SETS_FILE,
    SUMMARY_JSON_FILE,
    DELTA_FILE,
    MANIFEST_FILE,
]:
    print("-", p)


Mounted at /content/drive
[Bootstrap] Atualizando repositório (git pull)...
[Bootstrap] CWD = /content/drive/MyDrive/Mestrado/PPCOMP_DM
DRIVE_ROOT = /content/drive/MyDrive/Mestrado
FULL_MODEL_FACING_PARQUET = /content/drive/MyDrive/Mestrado/02-datasets/99-full/03-model-facing/window_5min_series_allcells_model_facing_000000000000.parquet
STAGE04_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/04_FULL_episodes
STAGE05_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features
AGGREGATE_DIR = /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate
ACTIVE_CELLS = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
RAW_CORE = ['fail_rate', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'event_LOST_count']
TEMPORAL_CORE = ['fail_rate', 'n_events', 'n_failed', 'n_machines', 'n_collections', 'event_LOST_count', 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_1h', 'rolling_std_1h', 'pct_change', 'zscore_expanding']
[NB05_FULL_

,cell_id,rows_input,rows_output,rows_removed_trim,critical_windows_input,critical_windows_output,episodes_detected_input,empty_windows,fail_rate_max_abs_diff_recomputed,n_temporal_core,status
0,a,8928,8925,3,441,441,191,0,0.0,13,OK
1,b,8928,8925,3,478,478,219,0,0.0,13,OK
2,c,8928,8925,3,380,380,178,0,0.0,13,OK
3,d,8928,8925,3,413,413,130,0,0.0,13,OK
4,e,8928,8925,3,245,245,110,0,0.0,13,OK
5,f,8928,8925,3,247,247,135,0,0.0,13,OK
6,g,8928,8925,3,602,602,295,0,0.0,13,OK
7,h,8928,8925,3,293,293,225,0,0.0,13,OK



[Totais]


,n_cells_processed,rows_input_total,rows_output_total,rows_removed_trim_total,critical_windows_input_total,critical_windows_output_total,episodes_detected_total,empty_windows_total,n_manifest_files
0,8,71424,71400,24,3099,3099,1483,0,21



[Head aggregate features]


,scenario_label,cell_id,bucket_id,bucket_start_us,bucket_start_ts,n_events,n_events_lifecycle,n_failed,n_update_pending_running,n_machines,...,diagnostic_global_m2s_threshold,is_critical_global_diagnostic,is_empty_window,lag_1,lag_2,lag_3,rolling_mean_1h,rolling_std_1h,pct_change,zscore_expanding
0,W5_K24_H12_P1_TRAIN_M2S_FULL,a,5,1500000000,1970-01-01 00:25:00,71594,53352,1061,18242,8047,...,0.029038,0,0,0.014825,0.011300,0.011302,0.013062,0.001761,-0.000391,1.410726
1,W5_K24_H12_P1_TRAIN_M2S_FULL,a,6,1800000000,1970-01-01 00:30:00,90785,67785,1230,23000,8670,...,0.029038,0,0,0.014820,0.014825,0.011300,0.013159,0.001587,-0.085777,0.276380
2,W5_K24_H12_P1_TRAIN_M2S_FULL,a,7,2100000000,1970-01-01 00:35:00,124379,100670,1052,23709,8292,...,0.029038,0,0,0.013548,0.014820,0.014825,0.012376,0.002273,-0.375723,-2.962662
3,W5_K24_H12_P1_TRAIN_M2S_FULL,a,8,2400000000,1970-01-01 00:40:00,94720,83073,1007,11647,7228,...,0.029038,0,0,0.008458,0.013548,0.014820,0.012126,0.002191,0.256953,-0.767313
4,W5_K24_H12_P1_TRAIN_M2S_FULL,a,9,2700000000,1970-01-01 00:45:00,114738,91440,937,23298,8120,...,0.029038,0,0,0.010631,0.008458,0.013548,0.011631,0.002433,-0.231853,-1.807092



Artefatos aggregate:
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_feature_summary_by_cell.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_window_5min_features_active_cells.parquet
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_feature_sets.json
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_feature_engineering_summary.json
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_delta_vs_canonical.csv
- /content/drive/MyDrive/Mestrado/04-reports/99_FULL_downstream/05_FULL_features/aggregate/05_FULL_artifact_manifest_sha256.csv


## 10. Conclusão da etapa

O NB05_FULL concluiu com sucesso a engenharia de atributos temporais do experimento FULL, processando separadamente as oito células e preservando a ordem temporal de cada unidade experimental. Todas as células encerraram com `status = OK`, sem falhas nas travas metodológicas ou de integridade.

A entrada total compreendeu **71.424 janelas**, correspondentes a **8.928 janelas por célula**. A construção de `lag_1`, `lag_2` e `lag_3` exigiu a remoção das três observações iniciais de cada série, totalizando **24 linhas removidas**. O conjunto agregado de saída contém, portanto, **71.400 janelas**, ou **8.925 por célula**. Essa redução decorre exclusivamente da ausência de histórico suficiente para as três defasagens iniciais e não representa perda de episódios nem alteração da definição de criticidade.

A consistência dos dados de origem foi confirmada em todas as células. A recomputação de `fail_rate` a partir de `n_failed / n_events` apresentou diferença absoluta máxima igual a **0,0**. Também foi confirmada a igualdade exata entre `n_failed` e `event_FAIL_count`. A segunda coluna permanece nos Parquets como informação de proveniência, mas foi excluída dos vetores de preditores, evitando redundância determinística na modelagem.

Os conjuntos ativos ficaram definidos como:

- `raw_core`: **6 atributos** — `fail_rate`, `n_events`, `n_failed`, `n_machines`, `n_collections` e `event_LOST_count`;
- `temporal_core`: **13 atributos** — os seis atributos de `raw_core`, acrescidos de `lag_1`, `lag_2`, `lag_3`, `rolling_mean_1h`, `rolling_std_1h`, `pct_change` e `zscore_expanding`.

As verificações de cardinalidade, unicidade de nomes, ausência de atributos proibidos, exclusão de `event_FAIL_count` dos preditores e inexistência de duplicidades exatas não justificadas foram aprovadas. Os atributos temporais foram calculados exclusivamente dentro de cada célula, sem mistura entre séries, e o agregado final foi formado apenas após a conclusão do processamento independente.

A etapa preservou integralmente a criticidade herdada do NB04_FULL: foram mantidas **3.099 janelas críticas** e **1.483 episódios**. Por célula, as quantidades de janelas críticas permaneceram em: a=441, b=478, c=380, d=413, e=245, f=247, g=602 e h=293. As quantidades de episódios de entrada foram: a=191, b=219, c=178, d=130, e=110, f=135, g=295 e h=225. Não houve janelas vazias nas oito séries.

Foram materializados os Parquets e summaries por célula, além dos seis artefatos agregados da etapa. O manifesto SHA-256 registrou **21 arquivos**, cobrindo os artefatos por célula e a consolidação agregada. A equivalência entre os arquivos individuais e o Parquet agregado foi verificada durante a execução.

Dessa forma, o NB05_FULL está **aprovado** e apto a alimentar o NB06_FULL. A interpretação científica da etapa é que a base FULL passa a dispor de atributos temporais causais e rastreáveis por célula, sem duplicidade da contagem de falhas nos preditores e sem alteração dos limiares, episódios ou estados definidos nas etapas anteriores.
